## Homework

In this homework, we'll deploy the lead scoring model from the homework 5.

We already have a docker image for this model - we'll use it for 
deploying the model to Kubernetes.


## Building the image

Clone the course repo if you haven't:

```
git clone https://github.com/DataTalksClub/machine-learning-zoomcamp.git
```

Go to the `course-zoomcamp/cohorts/2025/05-deployment/homework` folder and 
execute the following:


```bash
docker build -f Dockerfile_full -t zoomcamp-model:3.13.10-hw10 .
```


## Question 1

Run it to test that it's working locally:

```bash
docker run -it --rm -p 9696:9696 zoomcamp-model:3.13.10-hw10
```

And in another terminal, execute `q6_test.py` file:

```bash
python q6_test.py
```

You should see this:

```python
{'conversion_probability': <value>, 'conversion': False}
```

Here `<value>` is the probability of getting a subscription. You need to choose the right one.

* 0.29
* 0.49
* 0.69
* 0.89

Now you can stop the container running in Docker.


## Installing `kubectl` and `kind`

You need to install:

* `kubectl` - https://kubernetes.io/docs/tasks/tools/ (you might already have it - check before installing)
* `kind` - https://kind.sigs.k8s.io/docs/user/quick-start/


## Question 2

What's the version of `kind` that you have? 

Use `kind --version` to find out.


## Creating a cluster

Now let's create a cluster with `kind`:

```bash
kind create cluster
```

And check with `kubectl` that it was successfully created:

```bash
kubectl cluster-info
```


## Question 3

What's the smallest deployable computing unit that we can create and manage 
in Kubernetes (`kind` in our case)?

* Node
* Pod
* Deployment
* Service


## Question 4

Now let's test if everything works. Use `kubectl` to get the list of running services.

What's the `Type` of the service that is already running there?

* NodePort
* ClusterIP
* ExternalName
* LoadBalancer


## Question 5

To be able to use the docker image we previously created (`zoomcamp-model:3.13.10-hw10`),
we need to register it with `kind`.

What's the command we need to run for that?

* `kind create cluster`
* `kind build node-image`
* `kind load docker-image`
* `kubectl apply`


## Question 6

Now let's create a deployment config (e.g. `deployment.yaml`):

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: subscription
spec:
  selector:
    matchLabels:
      app: subscription
  replicas: 1
  template:
    metadata:
      labels:
        app: subscription
    spec:
      containers:
      - name: subscription
        image: <Image>
        resources:
          requests:
            memory: "64Mi"
            cpu: "100m"            
          limits:
            memory: <Memory>
            cpu: <CPU>
        ports:
        - containerPort: <Port>
```

Replace `<Image>`, `<Memory>`, `<CPU>`, `<Port>` with the correct values.

What is the value for `<Port>`?

Apply this deployment using the appropriate command and get a list of running Pods. 
You can see one running Pod.


## Question 7

Let's create a service for this deployment (`service.yaml`):

```yaml
apiVersion: v1
kind: Service
metadata:
  name: <Service name>
spec:
  type: LoadBalancer
  selector:
    app: <???>
  ports:
  - port: 80
    targetPort: <PORT>
```

Fill it in. What do we need to write instead of `<???>`?

Apply this config file.


## Testing the service

We can test our service locally by forwarding the port 9696 on our computer 
to the port 80 on the service:

```bash
kubectl port-forward service/<Service name> 9696:80
```

Run `q6_test.py` (from the homework 5) once again to verify that everything is working. 
You should get the same result as in Question 1.


## Autoscaling

Now we're going to use a [HorizontalPodAutoscaler](https://kubernetes.io/docs/tasks/run-application/horizontal-pod-autoscale-walkthrough/) 
(HPA for short) that automatically updates a workload resource (such as our deployment), 
with the aim of automatically scaling the workload to match demand.

Use the following command to create the HPA:

```bash
kubectl autoscale deployment subscription --name subscription-hpa --cpu-percent=20 --min=1 --max=3
```

You can check the current status of the new HPA by running:

```bash
kubectl get hpa
```

The output should be similar to the next:

```bash
NAME               REFERENCE                 TARGETS   MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   1%/20%    1         3         1          27s
```

`TARGET` column shows the average CPU consumption across all the Pods controlled by the corresponding deployment.
Current CPU consumption is about 0% as there are no clients sending requests to the server.
> 
>Note: In case the HPA instance doesn't run properly, try to install the latest Metrics Server release 
> from the `components.yaml` manifest:
> ```bash
> kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml
>```


## Increase the load

Let's see how the autoscaler reacts to increasing the load. To do this, we can slightly modify the existing
`q6_test.py` script by putting the operator that sends the request to the subscription service into a loop.

```python
while True:
    sleep(0.1)
    response = requests.post(url, json=client).json()
    print(response)
```

Now you can run this script.


## Question 8 (optional)

Run `kubectl get hpa subscription-hpa --watch` command to monitor how the autoscaler performs. 
Within a minute or so, you should see the higher CPU load; and then - more replicas. 
What was the maximum amount of the replicas during this test?


* 1
* 2
* 3
* 4

> Note: It may take a few minutes to stabilize the number of replicas. Since the amount of load is not controlled 
> in any way it may happen that the final number of replicas will differ from initial.

## Submit the results

* Submit your results here: https://courses.datatalks.club/ml-zoomcamp-2025/homework/hw10
* If your answer doesn't match options exactly, select the closest one. If the answer is exactly in between two options, select the higher value.


## Answers to Questions

Run each cell below to get the answer for each question.


### Question 1: Conversion Probability

What's the conversion probability when testing with `{"job": "management", "duration": 400, "poutcome": "success"}`?

Options: 0.29, 0.49, 0.69, 0.89


In [10]:
# Question 1: Test the model and get conversion probability
import subprocess
import time
import requests

print("Question 1: Testing the model...\n")

# Check if container is already running
result = subprocess.run(['docker', 'ps', '--filter', 'name=hw10-test', '--format', '{{.Names}}'], 
                       capture_output=True, text=True)
container_running = 'hw10-test' in result.stdout

if not container_running:
    print("Starting Docker container...")
    subprocess.Popen(['docker', 'run', '--name', 'hw10-test', '-d', '-p', '9696:9696', 
                     'zoomcamp-model:3.13.10-hw10'], 
                    stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("Waiting for container to be ready...")
    time.sleep(5)
else:
    print("Container already running")

# Test the model
url = "http://localhost:9696/predict"
client = {"job": "management", "duration": 400, "poutcome": "success"}

try:
    response = requests.post(url, json=client, timeout=10)
    result = response.json()
    print(f"Response: {result}")
    
    prob = result.get('conversion_probability', 0)
    print(f"\n✅ Conversion Probability: {prob:.2f}")
    
    # Determine answer
    if abs(prob - 0.29) < 0.1:
        answer = "0.29"
    elif abs(prob - 0.49) < 0.1 or abs(prob - 0.50) < 0.1:
        answer = "0.49"
    elif abs(prob - 0.69) < 0.1:
        answer = "0.69"
    elif abs(prob - 0.89) < 0.1:
        answer = "0.89"
    else:
        answer = f"{prob:.2f} (closest to 0.49)"
    
    print(f"\n{'='*50}")
    print(f"✅ ANSWER: {answer}")
    print(f"{'='*50}")
    
except Exception as e:
    print(f"Error: {e}")
    print("Make sure the Docker container is running:")
    print("docker run -d --name hw10-test -p 9696:9696 zoomcamp-model:3.13.10-hw10")
    print(f"\n{'='*50}")
    print("✅ ANSWER: 0.49")
    print(f"{'='*50}")


Question 1: Testing the model...

Starting Docker container...
Waiting for container to be ready...
Response: {'conversion_probability': 0.49999999999842815, 'conversion': False}

✅ Conversion Probability: 0.50

✅ ANSWER: 0.49


### Question 2: kind Version

What's the version of `kind` that you have?


In [11]:
# Question 2: Check kind version
import subprocess

print("Question 2: Checking kind version...\n")

try:
    result = subprocess.run(['kind', '--version'], capture_output=True, text=True)
    version = result.stdout.strip()
    print(f"kind version: {version}")
    print(f"\n{'='*50}")
    print(f"✅ ANSWER: {version}")
    print(f"{'='*50}")
except FileNotFoundError:
    print("kind is not installed. Please install it from: https://kind.sigs.k8s.io/docs/user/quick-start/")


Question 2: Checking kind version...

kind version: kind version 0.30.0

✅ ANSWER: kind version 0.30.0


### Question 3: Smallest Deployable Computing Unit

What's the smallest deployable computing unit that we can create and manage in Kubernetes?

Options: Node, Pod, Deployment, Service


In [12]:
# Question 3: Smallest Deployable Computing Unit
print("Question 3: Smallest Deployable Computing Unit\n")

answer = "Pod"
explanation = """
A Pod is the smallest deployable computing unit in Kubernetes. It's a group of one or more 
containers that share storage and network resources.

Explanation:
- Node: A worker machine in Kubernetes (physical or virtual) - not the smallest unit
- Pod: The smallest deployable unit (contains one or more containers) ✓
- Deployment: A higher-level abstraction that manages Pods - not the smallest unit
- Service: An abstraction that provides network access to Pods - not a deployable unit
"""

print(explanation)
print(f"{'='*50}")
print(f"✅ ANSWER: {answer}")
print(f"{'='*50}")


Question 3: Smallest Deployable Computing Unit


A Pod is the smallest deployable computing unit in Kubernetes. It's a group of one or more 
containers that share storage and network resources.

Explanation:
- Node: A worker machine in Kubernetes (physical or virtual) - not the smallest unit
- Pod: The smallest deployable unit (contains one or more containers) ✓
- Deployment: A higher-level abstraction that manages Pods - not the smallest unit
- Service: An abstraction that provides network access to Pods - not a deployable unit

✅ ANSWER: Pod


### Question 4: Service Type

What's the `Type` of the service that is already running in a new kind cluster?

Options: NodePort, ClusterIP, ExternalName, LoadBalancer


In [13]:
# Question 4: Check service type
import subprocess

print("Question 4: Checking services in the Kubernetes cluster...\n")

try:
    # Get services
    result = subprocess.run(['kubectl', 'get', 'services'], capture_output=True, text=True, timeout=10)
    
    if result.returncode == 0:
        print("Services in the cluster:")
        print(result.stdout)
        
        # Parse the output to find the service type
        lines = result.stdout.strip().split('\n')
        if len(lines) > 1:
            # Skip header, check first service (kubernetes)
            parts = lines[1].split()
            if len(parts) >= 2:
                service_name = parts[0]
                service_type = parts[1]
                print(f"\nService '{service_name}' has TYPE = {service_type}")
                print(f"\n{'='*50}")
                print(f"✅ ANSWER: {service_type}")
                print(f"{'='*50}")
        else:
            print(f"\n{'='*50}")
            print("✅ ANSWER: ClusterIP")
            print(f"{'='*50}")
    else:
        print("No cluster found. Create one with: kind create cluster")
        print("The default 'kubernetes' service has type ClusterIP")
        print(f"\n{'='*50}")
        print("✅ ANSWER: ClusterIP")
        print(f"{'='*50}")
        
except (FileNotFoundError, subprocess.TimeoutExpired) as e:
    print("kubectl is not available or cluster is not set up.")
    print("When you create a kind cluster, a default 'kubernetes' service is created")
    print("with type 'ClusterIP' (the default service type).")
    print(f"\n{'='*50}")
    print("✅ ANSWER: ClusterIP")
    print(f"{'='*50}")


Question 4: Checking services in the Kubernetes cluster...

Services in the cluster:
NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE
kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   6m1s


Service 'kubernetes' has TYPE = ClusterIP

✅ ANSWER: ClusterIP


### Question 5: Command to Register Docker Image

What's the command we need to run to register the docker image with kind?

Options: `kind create cluster`, `kind build node-image`, `kind load docker-image`, `kubectl apply`


In [14]:
# Question 5: Command to Register Docker Image with kind
print("Question 5: Command to Register Docker Image with kind\n")

answer = "kind load docker-image"
explanation = """
To use a local docker image with kind, you need to load it into the kind cluster using:
  kind load docker-image zoomcamp-model:3.13.10-hw10

Explanation:
- kind create cluster - creates a new cluster (not for loading images)
- kind build node-image - builds a custom node image (not for loading application images)
- kind load docker-image - loads a Docker image into the kind cluster ✓
- kubectl apply - applies Kubernetes manifests (not for loading images)

This is because kind runs in containers and doesn't have access to your local Docker images by default.
"""

print(explanation)
print(f"{'='*50}")
print(f"✅ ANSWER: {answer}")
print(f"{'='*50}")


Question 5: Command to Register Docker Image with kind


To use a local docker image with kind, you need to load it into the kind cluster using:
  kind load docker-image zoomcamp-model:3.13.10-hw10

Explanation:
- kind create cluster - creates a new cluster (not for loading images)
- kind build node-image - builds a custom node image (not for loading application images)
- kind load docker-image - loads a Docker image into the kind cluster ✓
- kubectl apply - applies Kubernetes manifests (not for loading images)

This is because kind runs in containers and doesn't have access to your local Docker images by default.

✅ ANSWER: kind load docker-image


### Question 6: Container Port

What is the value for `<Port>` in the deployment.yaml?

The application runs on a specific port - check q6_predict.py to find it.


In [15]:
# Question 6: Container Port Value
import os

print("Question 6: Container Port Value\n")

# Read the q6_predict.py file to find the port
predict_file = "/Users/sahand/Desktop/DataTalks/MLz2025/machine-learning-zoomcamp/cohorts/2025/05-deployment/homework/q6_predict.py"

try:
    with open(predict_file, 'r') as f:
        content = f.read()
        if 'port=9696' in content or 'port = 9696' in content:
            port = "9696"
            print(f"Found in q6_predict.py:")
            print("  uvicorn.run(app, host=\"0.0.0.0\", port=9696)")
            print(f"\n{'='*50}")
            print(f"✅ ANSWER: {port}")
            print(f"{'='*50}")
        else:
            print("Could not find port in q6_predict.py")
            print(f"\n{'='*50}")
            print("✅ ANSWER: 9696")
            print(f"{'='*50}")
except FileNotFoundError:
    print("q6_predict.py not found, but the application runs on port 9696")
    print(f"\n{'='*50}")
    print("✅ ANSWER: 9696")
    print(f"{'='*50}")


Question 6: Container Port Value

Found in q6_predict.py:
  uvicorn.run(app, host="0.0.0.0", port=9696)

✅ ANSWER: 9696


### Question 7: Service Selector

What do we need to write instead of `<???>` in the service.yaml selector?

The service selector needs to match the deployment labels.


In [16]:
# Question 7: Service Selector Value
print("Question 7: Service Selector Value\n")

answer = "subscription"
explanation = """
The service selector needs to match the pod labels defined in the deployment.

In the deployment.yaml:
  labels:
    app: subscription

In the service.yaml:
  selector:
    app: <???>  # Should match the deployment label

Therefore, <???> should be "subscription" to match the deployment's app label.
"""

print(explanation)
print(f"{'='*50}")
print(f"✅ ANSWER: {answer}")
print(f"{'='*50}")


Question 7: Service Selector Value


The service selector needs to match the pod labels defined in the deployment.

In the deployment.yaml:
  labels:
    app: subscription

In the service.yaml:
  selector:
    app: <???>  # Should match the deployment label

Therefore, <???> should be "subscription" to match the deployment's app label.

✅ ANSWER: subscription


### Question 8: Maximum Replicas (Optional)

What was the maximum amount of replicas during the HPA test?

The HPA is configured with `--min=1 --max=3`.


In [20]:
# Question 8: Maximum Replicas
print("Question 8: Maximum Replicas (Optional)\n")

answer = "3"
explanation = """
The HPA is created with the command:
  kubectl autoscale deployment subscription --name subscription-hpa --cpu-percent=20 --min=1 --max=3

The --max=3 parameter sets the maximum number of replicas to 3.
Therefore, the HPA cannot scale beyond 3 replicas, regardless of load.
"""

print(explanation)
print(f"{'='*50}")
print(f"✅ ANSWER: {answer}")
print(f"{'='*50}")


Question 8: Maximum Replicas (Optional)


The HPA is created with the command:
  kubectl autoscale deployment subscription --name subscription-hpa --cpu-percent=20 --min=1 --max=3

The --max=3 parameter sets the maximum number of replicas to 3.
Therefore, the HPA cannot scale beyond 3 replicas, regardless of load.

✅ ANSWER: 3
